In [ ]:
import pandas as pd
import geopandas as gpd
from difflib import SequenceMatcher
import re
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv(r"datasets/final_clustering.csv")

In [ ]:
df.head()

,region_name,mo,23-01,23-02,23-03,23-04,23-05,23-06,23-07,23-08,...,24-05,24-06,24-07,24-08,24-09,24-10,24-11,24-12,cluster_2023,cluster_2024
0,Алтайский край,Алейский муниципальный район,NaN,NaN,NaN,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1,Алтайский край,Алтайский муниципальный район,1.0,1.0,1.0,0.0,1.0,3.0,3.0,3.0,...,1.0,0.0,0.0,1.0,1.0,1.0,3.0,1.0,3.0,3.0
2,Алтайский край,Баевский муниципальный район,1.0,1.0,1.0,1.0,1.0,1.0,4.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Алтайский край,Бийский муниципальный район,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,...,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
4,Алтайский край,Благовещенский муниципальный район,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0


In [ ]:
gdf = gpd.read_file("datasets/Municipalities_Russia_2025.geojson")

In [ ]:
gdf.head()

,region,type,name,admcen,fo,arctic,okato,oktmo,geometry
0,Алтайский край,муниципальный район,Алейский муниципальный район,г Алейск,Сибирский,0,01201000000,01601000000,"MULTIPOLYGON (((9261395.767 6873159.859, 92674..."
1,Алтайский край,муниципальный район,Алтайский муниципальный район,с Алтайское,Сибирский,0,01202000000,01602000000,"MULTIPOLYGON (((9551392.113 6809974.372, 95511..."
2,Алтайский край,муниципальный район,Баевский муниципальный район,с Баево,Сибирский,0,01203000000,01603000000,"MULTIPOLYGON (((9025684.894 7055484.739, 90236..."
3,Алтайский край,муниципальный район,Бийский муниципальный район,г Бийск,Сибирский,0,01204000000,01604000000,"MULTIPOLYGON (((9582010.127 6948721.29, 958356..."
4,Алтайский край,муниципальный район,Благовещенский муниципальный район,рп Благовещенка,Сибирский,0,01205000000,01605000000,"MULTIPOLYGON (((8971203.02 6997278.35, 8968938..."


In [ ]:
gdf_names = gdf['name'].dropna().astype(str).unique()

with open('datasets/gdf_names.txt', 'w', encoding='utf-8') as f:
    for n in sorted(gdf_names):
        f.write(str(n) + '\n')

df_names = df['mo'].dropna().astype(str).unique()

with open('datasets/df_names.txt', 'w', encoding='utf-8') as f:
    for n in sorted(df_names):
        f.write(str(n) + '\n')


Некоторые МО названы по-разному в двух датасетах. Нужно изменить названия мунициальных образований в gdf на те, которые есть в df. Составляем список таких образований

In [ ]:
with open('datasets/df_names.txt', encoding='utf-8') as f:
    df_list = [l.strip() for l in f if l.strip()]

with open('datasets/gdf_names.txt', encoding='utf-8') as f:
    gdf_list = [l.strip() for l in f if l.strip()]

print(f"df: {len(df_list)} названий")
print(f"gdf: {len(gdf_list)} названий")

def strip_type(name):
    s = name.lower().strip()
    s = re.sub(r'\b(муниципальный|муниципальное|городской|городское|сельское|сельский)\b', '', s)
    s = re.sub(r'\b(район|округ|поселение|город|г\.|пос\.|пгт)\b', '', s)
    s = re.sub(r'[^\w\s\-]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

rows = []
for d in df_list:
    core_d = strip_type(d)
    best_score, best_g = 0, None
    for g in gdf_list:
        score = SequenceMatcher(None, core_d, strip_type(g)).ratio()
        if score > best_score:
            best_score, best_g = score, g
    if best_score > 0.9:
        rows.append({'df_name': d, 'gdf_name': best_g, 'score': round(best_score, 2)})

pairs = pd.DataFrame(rows)
print(f"Найдено пар: {len(pairs)}")

blacklist = {
    ('Волжский муниципальный район', 'город Волжский'),
    ('Гусь-Хрустальный муниципальный район', 'город Гусь-Хрустальный'),
    ('Слободской муниципальный район', 'город Слободской'),
    ('Октябрьский муниципальный район', 'город Октябрьский'),
    ('Октябрьский муниципальный округ', 'город Октябрьский'),
    ('Троицкий муниципальный район', 'Троицкий'),
    ('Троицкий муниципальный район', 'Троицкий муниципальный округ'),
    ('Ленинский муниципальный район', 'Ленинский'),
    ('Ленинский муниципальный район', 'Ленинский муниципальный округ'),
    ('Советский муниципальный район', 'Советский'),
    ('Советский муниципальный округ', 'Советский'),
    ('Советский муниципальный район', 'Советский муниципальный округ'),
    ('Советский муниципальный округ', 'Советский муниципальный район'),
    ('Кировский муниципальный район', 'Кировский'),
    ('Кировский муниципальный округ', 'Кировский'),
    ('Кировский муниципальный район', 'Кировский муниципальный округ'),
    ('Кировский муниципальный округ', 'Кировский муниципальный район'),
    ('Петровский муниципальный район', 'Петровский'),
    ('Петровский муниципальный округ', 'Петровский'),
    ('Петровский муниципальный район', 'Петровский муниципальный округ'),
    ('Петровский муниципальный округ', 'Петровский муниципальный район'),
    ('Павловский муниципальный район', 'Павловский'),
    ('Павловский муниципальный округ', 'Павловский'),
    ('Павловский муниципальный район', 'Павловский муниципальный округ'),
    ('Павловский муниципальный округ', 'Павловский муниципальный район'),
    ('Первомайский муниципальный район', 'Первомайский'),
    ('Первомайский муниципальный округ', 'Первомайский'),
    ('Первомайский муниципальный район', 'Первомайский муниципальный округ'),
    ('Первомайский муниципальный округ', 'Первомайский муниципальный район'),
    ('Знаменский муниципальный район', 'Знаменский'),
    ('Знаменский муниципальный округ', 'Знаменский'),
    ('Знаменский муниципальный район', 'Знаменский муниципальный округ'),
    ('Знаменский муниципальный округ', 'Знаменский муниципальный район'),
    ('Никольский муниципальный район', 'Никольский'),
    ('Никольский муниципальный округ', 'Никольский'),
    ('Никольский муниципальный район', 'Никольский муниципальный округ'),
    ('Никольский муниципальный округ', 'Никольский муниципальный район'),
    ('Спасский муниципальный район', 'Спасский'),
    ('Спасский муниципальный округ', 'Спасский'),
    ('Спасский муниципальный район', 'Спасский муниципальный округ'),
    ('Спасский муниципальный округ', 'Спасский муниципальный район'),
    ('Дмитровский муниципальный район', 'Дмитровский'),
    ('Дмитровский муниципальный округ', 'Дмитровский'),
    ('Дмитровский муниципальный район', 'Дмитровский муниципальный округ'),
    ('Беловский муниципальный округ', 'Беловский'),
    ('Беловский муниципальный округ', 'Беловский муниципальный район'),
    ('Кемеровский муниципальный округ', 'Кемеровский'),
    ('Новокузнецкий муниципальный округ', 'Новокузнецкий'),
    ('Прокопьевский муниципальный округ', 'Прокопьевский'),
    ('Сосновоборский муниципальный район', 'Сосновоборский'),
    ('Корсаковский муниципальный район', 'Корсаковский'),
    ('Чебаркульский муниципальный район', 'Чебаркульский'),
    ('Камышловский муниципальный район', 'Камышловский'),
    ('Чусовской муниципальный округ', 'Чусовской муниципальный округ'),
    ('Ильинский муниципальный район', 'Ильинский муниципальный округ'),
    ('Партизанский муниципальный район', 'Партизанский'),
    ('Партизанский муниципальный район', 'Партизанский муниципальный округ'),
    ('Пермский муниципальный район', 'Пермский'),
    ('Добрянский муниципальный район', 'Добрянский муниципальный округ'),
    ('Ленинск-Кузнецкий муниципальный округ', 'Ленинск-Кузнецкий муниципальный округ'),
}

pairs_clean = pairs[~pairs.apply(lambda r: (r['df_name'], r['gdf_name']) in blacklist, axis=1)].copy()
print(f"После чёрного списка: {len(pairs_clean)}")

def priority(row):
    d, g = row['df_name'], row['gdf_name']
    if 'район' in d and 'округ' in g:
        return 1
    if 'район' in d and 'округ' not in g and 'район' not in g:
        return 2
    if d.startswith('городской округ') and not g.startswith('городской округ'):
        return 3
    if 'округ' in d and 'район' in g:
        return 4
    return 5

pairs_clean['priority'] = pairs_clean.apply(priority, axis=1)
pairs_clean = pairs_clean.sort_values(['df_name', 'priority'])
pairs_final = pairs_clean.drop_duplicates(subset='df_name', keep='first')


pairs_final[['df_name', 'gdf_name']].to_csv(
    'replace_map_clean.csv', sep=';', index=False, encoding='utf-8'
)

print(f"\nИтог: {len(pairs_final)} уникальных пар")
print("Файл: replace_map_clean.csv")

df: 1054 названий
gdf: 2566 названий
Найдено пар: 1044
После чёрного списка: 1024

Итог: 1024 уникальных пар
Файл: replace_map_clean.csv


Заменяем названия в gdf и объединяем его с df

In [ ]:
rmap = pd.read_csv('replace_map_clean.csv', sep=';', encoding='utf-8')
rename_dict = dict(zip(rmap['gdf_name'], rmap['df_name']))

gdf['name_for_merge'] = gdf['name'].map(rename_dict).fillna(gdf['name'])

In [ ]:
rmap = pd.read_csv(
    'replace_map_clean.csv',
    sep=';',
    encoding='utf-8'
)

rename_dict = dict(
    zip(rmap['gdf_name'], rmap['df_name'])
)

gdf['name_for_merge'] = (
    gdf['name']
    .map(rename_dict)
    .fillna(gdf['name'])
)

cluster_map = (
    df[['mo', 'cluster_2024']]
    .dropna(subset=['cluster_2024'])
    .drop_duplicates('mo')
    .rename(columns={'cluster_2024': 'cluster'})
    .copy()
)

cluster_map['cluster'] = cluster_map['cluster'].astype(int)

print("Кластеры 2024:")
print(cluster_map['cluster'].value_counts().sort_index())

gdf_base = gdf[
    ['name_for_merge', 'geometry']
].copy()

merged = gdf_base.merge(
    cluster_map,
    left_on='name_for_merge',
    right_on='mo',
    how='left'
)

print(f"Территорий после merge: {len(merged)}")
print(merged.columns.tolist())

Кластеры 2024:
cluster
0    210
1    287
2    216
3    171
4    108
Name: count, dtype: int64
Территорий после merge: 2674
['name_for_merge', 'geometry', 'mo', 'cluster']


Упрощаем геометрию, чтобы файл с картой был легче, и красим карту России по кластерам

In [ ]:
merged = merged.reset_index(drop=True)
merged['id'] = merged.index.astype(str)

map_gdf = merged[
    ['id', 'name_for_merge', 'cluster', 'geometry']
].copy()

map_gdf = map_gdf.to_crs(epsg=6933)

map_gdf['geometry'] = map_gdf.geometry.simplify(
    5000,
    preserve_topology=True
)

map_gdf = map_gdf.to_crs(epsg=4326)
geojson_all = map_gdf.__geo_interface__

print(
    "Размер GeoJSON:",
    round(len(str(geojson_all)) / 1024 / 1024, 2),
    "МБ"
)
plot_df = map_gdf.copy()

plot_df['cluster_map'] = (
    plot_df['cluster']
    .fillna(-1)
    .astype(int)
)

fig = go.Figure()

fig.add_trace(
    go.Choroplethmap(
        geojson=geojson_all,
        locations=plot_df['id'],
        z=plot_df['cluster_map'],
        text=plot_df['name_for_merge'],

colorscale=[
    [0.000, '#FFFFFF'], [0.166, '#FFFFFF'],
    [0.166, '#E53935'], [0.333, '#E53935'],
    [0.333, '#FDD835'], [0.500, '#FDD835'],
    [0.500, '#43A047'], [0.666, '#43A047'],
    [0.666, '#1E88E5'], [0.833, '#1E88E5'],
    [0.833, '#FB8C00'], [1.000, '#FB8C00'],
],
zmin=-1,
zmax=5,

        showscale=False,

        marker_opacity=0.9,
        marker_line_width=0.5,
        marker_line_color='black',

        hovertemplate=(
            '<b>%{text}</b><br>'
            'Кластер: %{z}'
            '<extra></extra>'
        )
    )
)

fig.update_layout(
    map=dict(
    style='white-bg',
    center={
        'lat': 40,
        'lon': 165
    },
    zoom=1.8
),
    height=900,
    width=1400,
    margin=dict(l=0, r=0, t=60, b=0),
    title='Кластеры МО, 2024 год'
)

fig.show(renderer='browser')

Размер GeoJSON: 4.55 МБ


In [ ]:
fig.write_html("map_clusters_2024.html")